# CommonThings ? ti?p t?c l??t train b? ng?t
Ch?y l?n l??t c?c ?. Notebook ch? resume `last.pt` c?a `Object_Detection_Train`, kh?ng kh?i t?o model pretrained hay t? b?t ??u l??t m?i.

Checkpoint v? b?n dataset d?ng l?i ???c l?u tr?n Drive. Colab v?n c? th? gi?i h?n GPU; khi GPU kh? d?ng l?i, ch?y notebook n?y t? ??u ?? kh?i ph?c m?i tr??ng v? ti?p t?c t? epoch ?? l?u. Epoch ?ang ch?y d? ph?i ch?y l?i.

Ngu?n: [Ultralytics resume](https://docs.ultralytics.com/modes/train/#resuming-interrupted-trainings), [Colab limits](https://research.google.com/colaboratory/faq.html).


## ? 1 ? c?i l?i th? vi?n c?a runtime
Phi?n b?n m?c ??nh theo l??t train ?? ghi nh?n trong d? ?n. N?u checkpoint b?o kh?c phi?n b?n, s?a bi?n n?y cho kh?p r?i kh?i ??ng l?i runtime v? ch?y l?i c?c ?.

In [ ]:
import subprocess
import sys
ULTRALYTICS_VERSION = "8.4.183"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"])

## ? 2 ? k?t n?i Drive v? ki?m tra checkpoint
T? ho?n th?nh b??c ??ng nh?p/c?p quy?n Google n?u Colab y?u c?u. Kh?ng d?ng `best.pt` ?? ho?n t?t thay cho checkpoint ?ang train.

In [ ]:
from pathlib import Path
from google.colab import drive
import torch
import ultralytics
from ultralytics import YOLO

MOUNT_DIR = Path("/CV_Object_Detection/drive")
MOUNT_DIR.parent.mkdir(parents=True, exist_ok=True)
drive.mount(str(MOUNT_DIR))
DRIVE_DIR = MOUNT_DIR / "MyDrive/YOLO_CommonThings"
WORK_DIR = Path("/content/commonthings")
RUN_DIR = DRIVE_DIR / "runs/Object_Detection_Train"
LAST_PT = RUN_DIR / "weights/last.pt"
NDJSON_PATH = DRIVE_DIR / "commonthings.ndjson"
WORK_DIR.mkdir(parents=True, exist_ok=True)
assert LAST_PT.is_file(), f"Kh?ng th?y {LAST_PT}. D?ng, kh?ng train l?i t? ??u."

# Ch? ??c checkpoint do ch?nh b?n train; kh?ng d?ng file .pt kh?ng r? ngu?n.
checkpoint = torch.load(LAST_PT, map_location="cpu", weights_only=False)
saved_epoch = int(checkpoint.get("epoch", -1))
saved_args = checkpoint.get("train_args", {})
total_epochs = int(saved_args.get("epochs", 0))
print("Checkpoint:", LAST_PT)
print("Phi?n b?n checkpoint:", checkpoint.get("version"))
print("?? ho?n t?t epoch:", saved_epoch + 1, "/", total_epochs)
print("Ultralytics ?ang c?i:", ultralytics.__version__)
assert checkpoint.get("version") in (None, ultralytics.__version__), "Phi?n b?n kh?ng kh?p. S?a ULTRALYTICS_VERSION ? ? 1 theo checkpoint r?i kh?i ??ng l?i runtime."
assert saved_epoch >= 0 and checkpoint.get("optimizer") is not None, "Checkpoint ?? b? strip optimizer ho?c ?? k?t th?c. Kh?ng th? resume nguy?n l??t n?y; notebook d?ng ?? tr?nh train l?i t? ??u."
assert saved_epoch + 1 < total_epochs, "?? ?? s? epoch, kh?ng c? epoch ?? resume."
checkpoint_names = (checkpoint.get("ema") or checkpoint.get("model")).names
print("L?p:", checkpoint_names)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Ch?a c? GPU ? c? th? chu?n b? dataset tr??c; ? train s? d?ng.")
del checkpoint

## ? 3 ? kh?i ph?c ?nh v? nh?n
D?ng NDJSON ?? n?m tr?n Drive. L?n ??u c?n t?i ?nh n?u ch?a c? b?n dataset d? ph?ng; n?u URL h?t h?n th? ? n?y d?ng. Sau khi ki?m tra ?? ?nh/nh?n, l?u m?t b?n ZIP tr?n Drive ?? c?c phi?n sau kh?ng c?n t?i l?i t? URL. Gi? nguy?n split v? nh?n r?ng.

In [ ]:
import hashlib
import json
import os
import shutil
import zipfile
from collections import Counter
import yaml
from PIL import Image
from ultralytics.data.converter import convert_ndjson_to_yolo

assert NDJSON_PATH.is_file(), f"Kh?ng th?y {NDJSON_PATH}. ??t ??ng NDJSON c?a l??t train n?y v?o ???ng d?n tr?n."
raw = NDJSON_PATH.read_bytes()
dataset_sha = hashlib.sha256(raw).hexdigest()
records = [json.loads(line) for line in raw.decode("utf-8-sig").splitlines() if line.strip()]
metadata = next(r for r in records if r.get("type") == "dataset")
image_records = [r for r in records if r.get("type") == "image"]
assert metadata.get("task") == "detect"
expected_splits = Counter(r["split"] for r in image_records)
assert expected_splits["train"] and expected_splits["val"], "C?n split train/val s?n; kh?ng t? chia l?i."
expected_boxes = sum(len(r.get("annotations", {}).get("boxes", [])) for r in image_records)

def names_dict(names):
    return dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
assert names_dict(metadata["class_names"]) == names_dict(checkpoint_names), "Danh s?ch/th? t? l?p kh?c checkpoint."

# R?ng bu?c c?c l?n resume sau v?i c?ng m?t NDJSON.
binding_path = RUN_DIR / "resume_dataset.sha256"
if binding_path.exists():
    assert binding_path.read_text().strip() == dataset_sha, "NDJSON ?? thay ??i so v?i l?n resume tr??c. D?ng ?? kh?ng thay dataset gi?a l??t train."
else:
    print("L?n ??u: ch?a c? fingerprint t? l??t train g?c. Ch? c? th? ki?m tra l?p v? n?i dung NDJSON hi?n t?i, ch?a ch?ng minh ??y l? ??ng phi?n b?n ?? d?ng tr??c ??.")

backup_zip = DRIVE_DIR / f"commonthings_resume_{dataset_sha[:16]}.zip"
local_root = WORK_DIR / f"resume_{dataset_sha[:16]}"
if backup_zip.is_file():
    local_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(backup_zip) as archive:
        for name in archive.namelist():
            assert (local_root / name).resolve().is_relative_to(local_root.resolve()), "ZIP ch?a ???ng d?n ngo?i dataset."
        archive.extractall(local_root)
    DATA_YAML = local_root / "data.yaml"
    print("Kh?i ph?c dataset t? ZIP tr?n Drive.")
else:
    DATA_YAML = Path(await convert_ndjson_to_yolo(str(NDJSON_PATH), output_path=str(WORK_DIR / "converted")))
    local_root = DATA_YAML.parent

assert DATA_YAML.is_file()
config = yaml.safe_load(DATA_YAML.read_text())
assert names_dict(config["names"]) == names_dict(checkpoint_names)
# Dataset converter c? c?c split t??ng ??i; tr? l?i ??ng g?c runtime hi?n t?i.
config["path"] = str(local_root)
for split in expected_splits:
    config[split] = f"images/{split}"
DATA_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

from ultralytics.data.utils import IMG_FORMATS
actual_boxes = 0
actual_empty = 0
for split, expected_count in expected_splits.items():
    images = [p for p in (local_root / "images" / split).rglob("*") if p.is_file() and p.suffix[1:].lower() in IMG_FORMATS]
    assert len(images) == expected_count, f"{split}: {len(images)}/{expected_count} ?nh. D?ng."
    for image_path in images:
        with Image.open(image_path) as image:
            image.verify()
        relative = image_path.relative_to(local_root / "images" / split)
        label_path = (local_root / "labels" / split / relative).with_suffix(".txt")
        assert label_path.is_file(), f"Thi?u nh?n: {label_path}"
        lines = [line.split() for line in label_path.read_text().splitlines() if line.strip()]
        actual_boxes += len(lines)
        actual_empty += not lines
        for values in lines:
            assert len(values) == 5, f"Nh?n detect kh?ng h?p l?: {label_path}"
            class_id, x, y, w, h = map(float, values)
            assert class_id.is_integer() and int(class_id) in names_dict(checkpoint_names)
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
    print(f"{split}: {len(images)} ?nh, ?? c?p nh?n.")
assert actual_boxes == expected_boxes, f"S? box kh?c NDJSON: {actual_boxes}/{expected_boxes}"
print("T?ng box:", actual_boxes, "| ?nh nh?n r?ng:", actual_empty)
print("DATA_YAML:", DATA_YAML)

if not backup_zip.is_file():
    local_archive = shutil.make_archive(str(WORK_DIR / f"backup_{dataset_sha[:16]}"), "zip", root_dir=local_root)
    pending_zip = backup_zip.with_suffix(".zip.partial")
    shutil.copy2(local_archive, pending_zip)
    os.replace(pending_zip, backup_zip)
binding_path.write_text(dataset_sha + "\n")
print("B?n dataset d?ng l?i ?:", backup_zip)

## ? 4 ? resume, kh?ng kh?i t?o model m?i
Gi? model v? l?ch h?c c?a checkpoint. L?u v?o ??ng l??t train tr?n Drive. M?i epoch ho?n t?t s? c? `last.pt`; khi GPU ng?t gi?a epoch, ph?n epoch ?ang d? ch?a ???c l?u. Notebook kh?ng th? g? gi?i h?n GPU c?a Colab.

In [ ]:
import shutil
from datetime import datetime, timezone

assert torch.cuda.is_available(), "Colab ch?a c?p GPU. D?ng t?i ??y; kh?ng train CPU hay train l?i t? ??u. K?t n?i runtime GPU khi h?n m?c cho ph?p r?i ch?y l?i notebook."
assert DATA_YAML.is_file() and LAST_PT.is_file()
# Gi? m?t b?n checkpoint tr??c resume, ?? kh?ng m?t ?i?m kh?i ph?c hi?n t?i.
backup_dir = DRIVE_DIR / "resume_backups"
backup_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
checkpoint_backup = backup_dir / f"last_epoch{saved_epoch + 1}_{stamp}.pt"
shutil.copy2(LAST_PT, checkpoint_backup)
assert checkpoint_backup.stat().st_size == LAST_PT.stat().st_size
print("Checkpoint d? ph?ng:", checkpoint_backup)
print(f"S? ti?p t?c epoch {saved_epoch + 2}/{total_epochs}, kh?ng b?t ??u epoch 1.")

model = YOLO(str(LAST_PT))
def verify_save_dir(trainer):
    assert Path(trainer.save_dir).resolve() == RUN_DIR.resolve(), "Th? m?c l?u kh?ng ??ng Drive. D?ng tr??c training."
    assert trainer.start_epoch == saved_epoch + 1, "Epoch resume kh?ng kh?p checkpoint."
    print("?? x?c nh?n resume v? l?u checkpoint tr?n Drive:", trainer.last)
model.add_callback("on_train_start", verify_save_dir)
model.train(
    resume=True,
    data=str(DATA_YAML),
    device=0,
    workers=2,
    save_dir=str(RUN_DIR),
)
print("Best:", RUN_DIR / "weights/best.pt")
print("Last:", RUN_DIR / "weights/last.pt")